In [ ]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
print(housing.data.shape, housing.target.shape)

(20640, 8) (20640,)


In [ ]:
import pandas as pd

# Create a DataFrame from the California housing dataset
housing_df = pd.DataFrame(housing.data, columns=housing.feature_names)
housing_df['MedHouseVal'] = housing.target

# Display the first 5 rows
display(housing_df.head())

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### Step 1: Data Cleanliness Check
We will check for null values, missing values, and analyze the distribution of the features to understand outliers.

In [ ]:
print("--- Missing Values Check ---")
print(housing_df.isnull().sum())

print("\n--- Dataset Statistics (to identify potential outliers) ---")
display(housing_df.describe())

--- Missing Values Check ---
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

--- Dataset Statistics (to identify potential outliers) ---


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


### Step 2: Build a Linear Regression Model
We will split the data into training and testing sets, train a Linear Regression model, and evaluate its performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Split features and target
X = housing_df.drop(columns=['MedHouseVal'])
y = housing_df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2) Score: {r2:.4f}")

# Save the trained model for our Streamlit dashboard
joblib.dump(model, 'linear_regression_model.pkl')

Mean Squared Error (MSE): 0.5559
R-squared (R2) Score: 0.5758


['linear_regression_model.pkl']

### Step 3: Set up the Streamlit Application & Ngrok Tunnel

In [ ]:
# Write the Streamlit application code to a local file
with open('app.py', 'w') as f:
    f.write('''import streamlit as st
import pandas as pd
import joblib
import numpy as np

st.title("California House Value Predictor")
st.write("Enter the details below to predict the median house value (in $100,000s):")

# Load the pre-trained model
model = joblib.load('linear_regression_model.pkl')

# User inputs for features
med_inc = st.number_input("Median Income (MedInc in $10k)", value=3.5, step=0.1)
house_age = st.number_input("Median House Age (HouseAge)", value=28.0, step=1.0)
ave_rooms = st.number_input("Average Rooms (AveRooms)", value=5.2, step=0.1)
ave_bedrms = st.number_input("Average Bedrooms (AveBedrms)", value=1.1, step=0.1)
population = st.number_input("Population", value=1425.0, step=10.0)
ave_occup = st.number_input("Average Occupancy (AveOccup)", value=3.0, step=0.1)
latitude = st.number_input("Latitude", value=35.6, step=0.1)
longitude = st.number_input("Longitude", value=-119.5, step=0.1)

input_data = np.array([[med_inc, house_age, ave_rooms, ave_bedrms, population, ave_occup, latitude, longitude]])

if st.button("Predict House Value"):
    prediction = model.predict(input_data)[0]
    st.success(f"Predicted Median House Value: ${prediction * 100000:,.2f} (${prediction:.4f} in $100,000s)")
''')

In [ ]:
# Install streamlit and pyngrok
!pip install pyngrok streamlit --quiet

import os
import subprocess
import time
from pyngrok import ngrok

# --- INSERT YOUR NGROK AUTH TOKEN HERE ---
# Get a free token from https://dashboard.ngrok.com/
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Terminate any existing background tunnels
ngrok.kill()

# Start Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)

# Open ngrok tunnel in the background
try:
    tunnel = ngrok.connect(addr='8501', proto='http')
    print(f"\nStreamlit Dashboard is successfully running in the background!")
    print(f"Access your dashboard here: {tunnel.public_url}\n")
except Exception as e:
    print(f"Error starting ngrok. Did you replace 'YOUR_NGROK_AUTH_TOKEN_HERE' with your real token?\nError details: {e}")

ERROR:pyngrok.process.ngrok:t=2026-10-19T11:51:24+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


Error starting ngrok. Did you replace 'YOUR_NGROK_AUTH_TOKEN_HERE' with your real token?
Error details: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import subprocess
import time

# Securely retrieve ngrok auth token from Colab Secrets
try:
    auth_token = userdata.get('NGROK_AUTH_TOKEN')
    ngrok.set_auth_token(auth_token)

    # Kill any existing sessions
    ngrok.kill()

    # Ensure Streamlit is running in the background
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
    time.sleep(3)

    # Open the tunnel
    tunnel = ngrok.connect(addr='8501', proto='http')
    print(f"\nStreamlit Dashboard is successfully running!")
    print(f"Access your dashboard here: {tunnel.public_url}\n")
except Exception as e:
    print(f"Could not start ngrok tunnel. Please ensure you have added NGROK_AUTH_TOKEN to your Colab secrets.\nError: {e}")


Streamlit Dashboard is successfully running!
Access your dashboard here: https://dictate-dandelion-bulginess.ngrok-free.dev



### Stopping and Restarting Streamlit
Run this cell to stop any running background Streamlit processes and launch a fresh instance.

In [ ]:
import subprocess
import os
import time

# --- STEP 1: STOP STREAMLIT ---
print("Stopping running Streamlit instances...")
# Find and kill any processes running streamlit
try:
    # Using pkill to terminate streamlit processes
    subprocess.run(["pkill", "-f", "streamlit"])
    print("Streamlit server successfully stopped.")
except Exception as e:
    print(f"No active Streamlit process found to stop or error occurred: {e}")

time.sleep(2)

Stopping running Streamlit instances...
Streamlit server successfully stopped.


In [ ]:
# --- STEP 2: RESTART STREAMLIT ---
print("\nRestarting Streamlit on port 8501...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)
print("Streamlit has been restarted! You can now access it using your existing ngrok link.")


Restarting Streamlit on port 8501...
Streamlit has been restarted! You can now access it using your existing ngrok link.
